# Paper figures

Builds every figure and table used in the paper from **saved** experiment outputs. No simulation runs here: rerun `run_mpc.ipynb` / `run_irf.ipynb` first, then run all cells. Outputs go to `data/output_data/paper-figures/`.

| # | Output | Files |
|---|---|---|
| 1 | Impact MPC by income, liquidity, borrowing, balance-sheet group, constraint | `paper_impact_mpc.*` |
| 2 | Aggregate MPC table and cumulative MPC chart | `paper_mpc_table.*`, `paper_cumulative_mpc.*` |
| 3 | MPC by the 8 balance-sheet groups; KVW robustness | `paper_mpc_by_group8.*`, `paper_mpc_by_kvw_class.csv` |
| 4 | IRFs: monetary policy, one-off transfer | `paper_irf_macro.*` |
| 5 | Transfer fiscal multiplier | `paper_transfer_multiplier.*` |

Excluded until fixed: income-tax IRF (payroll wedge), government consumption (rationing), income-uncertainty IRF (pending rerun with the +1% shock).

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.household_classes import (
    GROUP_LABELS,
    HTM_CLASSES,
    classify_balance_sheet_groups,
    classify_households,
)
from src.mpc_paper_groups import prepare_paper_groups
from src.paper_plots import GROUP_SHORT, cumulative_mpc_figure, irf_figure, mpc_impact_figure, multiplier_figure
from src.paper_plots import STYLE as paper_style
from src.visual_helpers import _prepare_irf_plot_panel

MPC_DIR = Path("data/output_data/mpc-experiment/analysis")
IRF_DIR = Path("data/output_data/irf-experiment/analysis")
OUT = Path("data/output_data/paper-figures")
OUT.mkdir(parents=True, exist_ok=True)
SHOW_TRANSFER_WEIGHTED_MEAN = False  # equals the household mean: transfers are equal within a seed

# MPC measure used in every paper figure and table.
# Cash MPC = change in household cash consumption spending, i.e. goods INCLUDING VAT (+ cash rent,
# unchanged on impact), deflated by CPI, per unit of real transfer. The transfer is cash, so the
# numerator must be cash too. The VAT-exclusive goods measure (real_mpc_impact) is lower by the
# factor 1/(1+VAT) = 1/1.13: a household spending the whole transfer on goods would show 0.885.
MPC_IMPACT = "cash_real_mpc_impact"
MPC_CUMULATIVE = {
    "2 quarters": "cash_real_cmpc_2p",
    "3 quarters": "cash_real_cmpc_3p",
    "4 quarters": "cash_real_cmpc_4q",
}

## 0. Load the MPC panel and classify households
Groups are measured in the **baseline run (no transfer) in the transfer quarter, before the transfer is paid** (`pre_shock_row`). Income and asset quintiles pool all seeds; the 8 balance-sheet groups split at the within-seed median of NLA/y, IFA/y and HA/y (High = strictly above), as in the calibration notebook and Table `tab:household_groups`. NLA = liquid financial assets − all debt.

In [ ]:
panel_path = MPC_DIR / "household_mpc_panel_filtered.csv"
if not panel_path.exists():
    panel_path = MPC_DIR / "household_mpc_panel.csv"
mpc_panel = pd.read_csv(panel_path)
mpc_panel = mpc_panel.join(classify_balance_sheet_groups(mpc_panel)[["group_code", "group_label"]])
mpc_panel, specs = prepare_paper_groups(mpc_panel)
spec = {column: (column, title, order) for column, title, order in specs}
print(f"{len(mpc_panel):,} household-seed observations, {mpc_panel.seed.nunique()} seeds")

## 1. Impact MPC
**Measure: cash MPC** (`cash_real_mpc_impact`): real change in cash consumption spending (goods including VAT) in the transfer quarter per unit of real transfer. Boxes: 5th–95th percentile whiskers, dots: household means.

Panels: household income, liquid financial assets, net liquid assets (all debt), borrowing; balance-sheet groups (warm = low net liquid assets, i.e. hand-to-mouth side; cool = high) and the consumption liquidity constraint.

In [ ]:
top_row = [
    spec["income_paper_group"][:1] + ("Household income",) + (["Q1", "Q2", "Q3", "Q4", "Q5"],),
    spec["liquid_financial_assets_paper_group"][:1] + ("Liquid financial assets",) + (["Q1", "Q2", "Q3", "Q4", "Q5"],),
    spec["net_liquid_financial_assets_all_debt_paper_group"][:1]
    + ("Net liquid assets",)
    + (["Q1", "Q2", "Q3", "Q4", "Q5"],),
    ("borrowing", "Borrowing", ["No borrowing", "Consumer\ncredit", "Mortgage\nonly", "Both"]),
    ("constraint", "Liquidity constraint", ["Constrained", "Unconstrained"]),
]
impact_means = mpc_impact_figure(mpc_panel, outcome=MPC_IMPACT, group_specs=top_row, path=OUT / "paper_impact_mpc")
impact_means.to_csv(OUT / "paper_impact_mpc_group_means.csv", index=False)

## 2. Aggregate MPC table
Cash MPC, transfer-weighted (ΣΔC / Σtransfer) over **all** households in the filtered panel, for impact and cumulative horizons. No labour-income identification filter: cumulative MPCs include the general-equilibrium feedback of the transfer on labour income (net about +1% of the transfer). SE: standard deviation of per-seed MPCs / √seeds.

In [3]:
def weighted_mpc(panel, column):
    d = panel[np.isfinite(panel[column]) & np.isfinite(panel.real_shock_amount)]
    w = d.real_shock_amount
    per_seed = (d[column] * w).groupby(d.seed).sum() / w.groupby(d.seed).sum()
    return pd.Series(
        {
            "MPC": float((d[column] * w).sum() / w.sum()),
            "SE (seed)": float(per_seed.std(ddof=1) / np.sqrt(len(per_seed))),
            "Share < 0": float((d[column] < 0).mean()),
            "N": int(len(d)),
            "Seeds": int(d.seed.nunique()),
        }
    )


mpc_table = pd.DataFrame(
    {"Impact (transfer quarter)": weighted_mpc(mpc_panel, MPC_IMPACT)}
    | {f"Cumulative, {k}": weighted_mpc(mpc_panel, v) for k, v in MPC_CUMULATIVE.items()}
).T.astype({"N": int, "Seeds": int})
mpc_table.to_csv(OUT / "paper_mpc_table.csv")
mpc_table.style.format(
    {"MPC": "{:.3f}", "SE (seed)": "{:.4f}", "Share < 0": "{:.1%}", "N": "{:,.0f}", "Seeds": "{:.0f}"}
)

,MPC,SE (seed),Share < 0,N,Seeds
Impact (transfer quarter),0.445,0.0027,0.3%,"137,624",30
"Cumulative, 2 quarters",0.626,0.0032,3.2%,"137,624",30
"Cumulative, 3 quarters",0.865,0.0161,9.0%,"137,624",30
"Cumulative, 4 quarters",0.956,0.0242,14.2%,"137,624",30


In [4]:
latex = mpc_table[["MPC", "SE (seed)", "Share < 0", "N"]].to_latex(
    float_format="%.3f", header=["MPC", "SE (seed)", "Share $<0$", "$N$"], escape=False
)
(OUT / "paper_mpc_table.tex").write_text(latex)
print(latex)

\begin{tabular}{lrrrr}
\toprule
 & MPC & SE (seed) & Share $<0$ & $N$ \\
\midrule
Impact (transfer quarter) & 0.445 & 0.003 & 0.003 & 137624 \\
Cumulative, 2 quarters & 0.626 & 0.003 & 0.032 & 137624 \\
Cumulative, 3 quarters & 0.865 & 0.016 & 0.090 & 137624 \\
Cumulative, 4 quarters & 0.956 & 0.024 & 0.142 & 137624 \\
\bottomrule
\end{tabular}



### Cumulative MPC chart

In [ ]:
horizons = {
    1: MPC_IMPACT,
    2: MPC_CUMULATIVE["2 quarters"],
    3: MPC_CUMULATIVE["3 quarters"],
    4: MPC_CUMULATIVE["4 quarters"],
}
aggregate = pd.DataFrame({q: weighted_mpc(mpc_panel, col)[["MPC", "SE (seed)"]] for q, col in horizons.items()}).T
aggregate.columns = ["MPC", "SE"]
by_group = pd.DataFrame(
    {
        code: {q: weighted_mpc(mpc_panel[mpc_panel.group_code == code], col)["MPC"] for q, col in horizons.items()}
        for code in GROUP_SHORT
    }
)
cumulative_mpc_figure(aggregate, by_group, path=OUT / "paper_cumulative_mpc")
by_group.round(3)

## 3. MPC by balance-sheet group (table)
All eight groups. KVW (2014) classes follow as a robustness check (monthly pay, one-month credit-limit proxy, DW = top two quintiles).

In [ ]:
fmt = {"Population share": "{:.1%}", "MPC": "{:.3f}", "SE (seed)": "{:.4f}", "Share < 0": "{:.1%}", "N": "{:,.0f}"}


def group_table(panel, column, order):
    rows = {
        g: weighted_mpc(panel[panel[column] == g], MPC_IMPACT).to_dict()
        | {"Population share": float((panel[column] == g).mean())}
        for g in order
    }
    return pd.DataFrame(rows).T[["Population share", "MPC", "SE (seed)", "Share < 0", "N"]].astype({"N": int})


eight_table = group_table(mpc_panel, "group_code", list(GROUP_LABELS))
eight_table.insert(0, "Group", [GROUP_LABELS[c] for c in eight_table.index])
eight_table.to_csv(OUT / "paper_mpc_by_group8.csv")
(OUT / "paper_mpc_by_group8.tex").write_text(eight_table.to_latex(float_format="%.3f", escape=False))
eight_table.style.format(fmt)

In [7]:
# 3c. Robustness: KVW classes (monthly pay, one-month credit-limit proxy, DW = top two quintiles).
mpc_panel["htm_class"] = classify_households(mpc_panel)
kvw_table = group_table(mpc_panel, "htm_class", HTM_CLASSES)
kvw_table.to_csv(OUT / "paper_mpc_by_kvw_class.csv")
kvw_table.style.format(fmt)

/var/folders/07/tlqxnjyj74l8z1_hldncmj200000gr/T/ipykernel_82876/262875081.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mpc_panel["htm_class"] = classify_households(mpc_panel)


,Population share,MPC,SE (seed),Share < 0,N
P-HtM,16.8%,0.550,0.0037,0.1%,"23,176"
R-HtM,34.9%,0.567,0.0037,0.3%,"47,999"
DW,24.7%,0.374,0.0030,0.4%,"33,939"
Other,23.6%,0.265,0.0022,0.5%,"32,510"


## 4. Impulse responses
Mean across seeds (line) and 10th–90th percentile across seeds (band); deviation from the paired no-shock run. % for real GDP, real consumption and CPI; percentage points for unemployment and the policy rate (**annualised**: the model stores a quarterly rate). Horizon 0 = first quarter with the shock.

In [ ]:
PAPER_IRF_SHOCKS = {
    "policy_rate_100bp": "Monetary policy",
    "household_transfer_10pct": "One-off transfer",
    "unemp_2pp": "Job losses",
    # Excluded until fixed: income_tax_1pp (payroll wedge), gov_cons_5pct (rationing),
    # total_income_uncertainty_1pct: add after the rerun if clean.
}
PAPER_IRF_VARIABLES = {
    "real_gdp": "Real GDP, %",
    "real_cons": "Real consumption, %",
    "cpi": "CPI, %",
    "unemp": "Unemployment, pp",
    "policy_rate": "Policy rate, pp (annual)",
    "consumption_uncertainty_contribution": "Uncertainty term in Δln C, %",
}
MAX_HORIZON = 16

irf_raw = pd.read_csv(IRF_DIR / "irf_panel.csv")
irf_raw = irf_raw[irf_raw.shock_name.isin(PAPER_IRF_SHOCKS)]
irf = _prepare_irf_plot_panel(irf_raw)
irf = irf[irf.variable.isin(PAPER_IRF_VARIABLES) & (irf.horizon <= MAX_HORIZON)]
bands = (
    irf.groupby(["shock_name", "variable", "horizon"])["plot_value"]
    .agg(mean="mean", p10=lambda x: x.quantile(0.1), p90=lambda x: x.quantile(0.9), n="count")
    .reset_index()
)
bands.to_csv(OUT / "paper_irf_bands.csv", index=False)


# The model's policy rate is per quarter: annualise for display.
rate = bands.variable == "policy_rate"
bands.loc[rate, ["mean", "p10", "p90"]] *= 4
bands.to_csv(OUT / "paper_irf_bands.csv", index=False)
irf_figure(bands, shocks=PAPER_IRF_SHOCKS, variables=PAPER_IRF_VARIABLES, path=OUT / "paper_irf_macro")
print("seeds per cell:", bands.n.unique())

In [9]:
# Peak responses for the text.
peaks = bands.loc[bands.groupby(["shock_name", "variable"])["mean"].apply(lambda s: s.abs().idxmax())]
peaks[["shock_name", "variable", "horizon", "mean", "p10", "p90"]].round(3)

,shock_name,variable,horizon,mean,p10,p90
1,household_transfer_10pct,cpi,1,0.267,0.229,0.320
19,household_transfer_10pct,policy_rate,2,0.007,0.002,0.012
34,household_transfer_10pct,real_cons,0,5.179,5.054,5.356
54,household_transfer_10pct,real_gdp,3,0.876,0.475,1.153
69,household_transfer_10pct,unemp,1,-1.013,-1.253,-0.833
92,policy_rate_100bp,cpi,7,-0.135,-0.241,-0.036
105,policy_rate_100bp,policy_rate,3,0.820,0.809,0.831
122,policy_rate_100bp,real_cons,3,-2.891,-3.733,-1.969
143,policy_rate_100bp,real_gdp,7,-1.132,-1.570,-0.768
160,policy_rate_100bp,unemp,7,1.061,0.584,1.443


### Job-loss shock: who pays for the lost wages?
Changes in household income components, in % of baseline household income, so the parts add up to the total. Needs a rerun of `run_irf.ipynb` that saves `household_employee_income`, `household_social_transfers` and `household_unemployment_benefits`.

In [ ]:
components = {
    "household_employee_income": "Employee income",
    "household_social_transfers": "Social transfers",
    "household_income": "Total household income",
}
raw_all = pd.read_csv(IRF_DIR / "irf_panel.csv")
job = raw_all[(raw_all.shock_name == "unemp_2pp") & (raw_all.horizon <= MAX_HORIZON)]
if not set(components).issubset(job.variable.unique()):
    print("Income components not in this IRF run: rerun run_irf.ipynb.")
else:
    base_income = job[job.variable == "household_income"].set_index(["seed", "horizon"]).baseline
    parts = (
        job[job.variable.isin(components)]
        .assign(
            share=lambda d: (
                100
                * d.delta.to_numpy()
                / base_income.reindex(pd.MultiIndex.from_frame(d[["seed", "horizon"]])).to_numpy()
            )
        )
        .groupby(["variable", "horizon"])
        .share.mean()
        .unstack(0)
    )
    with plt.rc_context(paper_style):
        fig, ax = plt.subplots(figsize=(5.5, 3.3))
        colors = {
            "household_employee_income": "#c0602b",
            "household_social_transfers": "#6f97bd",
            "household_income": "#1f3b5c",
        }
        for var, label in components.items():
            ax.plot(
                parts.index, parts[var], label=label, color=colors[var], lw=2.2 if var == "household_income" else 1.6
            )
        ax.axhline(0, color="#8a8a8a", lw=0.6)
        ax.set_xlabel("Quarters")
        ax.set_ylabel("% of baseline household income")
        ax.legend(fontsize=8)
        fig.savefig(OUT / "paper_job_loss_income.pdf")
        fig.savefig(OUT / "paper_job_loss_income.png", dpi=250)
        plt.show()
    parts.round(2)

## 5. Cumulative fiscal multiplier of the one-off transfer
$M_H = \sum_{h=0}^{H-1} \Delta Y^{real}_h \,/\, T$, per seed, then mean and SE across seeds.
- $T$: the transfer, measured as the h0 change in household income (labour incomes are identical across arms at h0).
- Real changes are in h0 prices: each arm is deflated by its own CPI relative to the baseline CPI at h0. No discounting.
- Baseline policy rate is at its floor: no monetary offset (favourable case for fiscal policy).

In [10]:
TRANSFER_SHOCK = "household_transfer_10pct"
MULTIPLIER_HORIZON = 16


def transfer_multipliers(irf_dir):
    """Per-quarter cumulative multipliers (mean and seed SE) for GDP and household consumption."""
    raw = pd.read_csv(irf_dir / "irf_panel.csv")
    raw = raw[raw.shock_name == TRANSFER_SHOCK]
    w = raw.pivot_table(index=["seed", "horizon"], columns="variable", values=["baseline", "shock", "delta"])
    cpi_b, cpi_s = w["baseline", "cpi"], w["shock", "cpi"]
    cpi_b0 = cpi_b.groupby(level=0).transform("first")
    transfer = w["delta", "household_income"].xs(0, level=1)
    out = {}
    for var, key in [("gdp_output", "gdp"), ("household_consumption", "cons")]:
        real = (w["shock", var] / (cpi_s / cpi_b0) - w["baseline", var] / (cpi_b / cpi_b0)).unstack()
        cum = real.cumsum(axis=1).div(transfer, axis=0).iloc[:, :MULTIPLIER_HORIZON]
        out[key] = cum.mean()
        out[f"{key}_se"] = cum.std(ddof=1) / np.sqrt(len(cum))
    curves = pd.DataFrame(out)
    curves.index = curves.index + 1  # quarters elapsed, including the transfer quarter
    share = (transfer / w["baseline", "household_income"].xs(0, level=1)).mean()
    return curves, share, len(transfer)


curves = {}
for label, irf_dir in {
    "Benchmark (rate at floor)": IRF_DIR,
    "No floor on the policy rate": IRF_DIR.parent.parent / "irf-experiment-no_elb" / "analysis",
}.items():
    if (irf_dir / "irf_panel.csv").exists():
        curves[label], share, n_seeds = transfer_multipliers(irf_dir)
        print(f"{label}: transfer = {share:.1%} of quarterly household income, {n_seeds} seeds")

multiplier_figure(curves, path=OUT / "paper_transfer_multiplier")
benchmark = curves["Benchmark (rate at floor)"]
multiplier = benchmark.loc[[1, 4, 8, 12, 16]].rename(
    columns={"gdp": "GDP", "gdp_se": "GDP SE", "cons": "Consumption", "cons_se": "Consumption SE"}
)
multiplier.index.name = "Quarters"
multiplier.to_csv(OUT / "paper_transfer_multiplier.csv")
(OUT / "paper_transfer_multiplier.tex").write_text(multiplier.to_latex(float_format="%.2f"))
multiplier.round(3)

Transfer = 8.6% of quarterly household income; seeds = 24


Multiplier  SE (seed)
Variable                          Quarters                       
GDP                               1              0.061      0.002
                                  4              0.561      0.022
                                  8              1.132      0.073
                                  12             1.429      0.131
                                  16             1.537      0.190
Household consumption (excl. VAT) 1              0.373      0.002
                                  4              0.811      0.022
                                  8              0.988      0.037
                                  12             1.076      0.062
                                  16             1.078      0.095